# Welcome to the Data Assimilation Cycling tutorial
***
In this tutorial we will explore the Local Ensemble Transform Kalman Filter (LETKF) in a cycling data assimilation framework using the quasi-geostrophic model. It is strongly recommended that you complete the single-cycle LETKF tutorial first, as we will build upon those concepts and focus on what is new about cycling. The LETKF cycling system sequentially estimates the atmospheric state by alternating between analysis and forecast steps over multiple cycles, allowing the ensemble to evolve with the dynamical system and observations over time. In the following we first provide an introduction to cycling concepts for DA, followed by data-assimilation cycling experiments.

## Introduction on Cycling

In practical atmospheric data assimilation systems, the analysis is not performed in isolation, but rather as part of a **cycling** process in which forecasts and analyses are continuously updated over time. In this framework, the analysis produced at one assimilation time serves as the initial condition for a short-range forecast, which in turn provides the background state for the next assimilation step, and so on. This repeated sequence of forecast → analysis → forecast forms the foundation of modern numerical weather prediction systems.
Within the Local Ensemble Transform Kalman Filter (LETKF), this cycling process is extended to an ensemble of model states, allowing both the atmospheric state and its uncertainty to evolve dynamically. At a given analysis time $t_k$, the ensemble (n=1..N) of background forecasts ${\{\mathbf{x}_b^n(t_k)\}_{n=1}^{N}}$ is used to estimate the flow-dependent background error covariance and to assimilate available observations. The LETKF produces an updated ensemble of analyses ${\{\mathbf{x}_a^{n}(t_k)\}}_{n=1}^{N}$, which are then integrated forward in time to the next analysis time $(t_{k+1})$:

<center>$ \mathbf{x}_b^{n}(t_{k+1}) = \mathcal{M}\big(\mathbf{x}_a^{n}(t_k)\big), $</center><br>


where $\mathcal{M}$ denotes the numerical model. In this way, the ensemble evolves through alternating forecast and analysis steps, continuously updating both the mean state and the associated uncertainty.
A key feature of LETKF cycling is that the background error covariance is propagated implicitly through the ensemble. Unlike static covariance models used in variational methods, the ensemble captures the growth, propagation, and deformation of forecast errors under the model dynamics. This allows the assimilation to remain responsive to evolving atmospheric conditions, with uncertainty naturally increasing in dynamically active regions and decreasing where the flow is more predictable.

However, the cycling framework also introduces several important challenges. Because the ensemble size is limited, repeated forecast–analysis cycles can lead to a gradual loss of ensemble spread, causing the system to become overconfident and potentially diverge from the true state. To counteract this, **covariance inflation** is typically applied at each cycle to maintain realistic uncertainty. At the same time, **covariance localization** remains essential to suppress spurious long-range correlations that arise from finite ensemble sampling. The choice of inflation method, localization radius, and assimilation frequency all play critical roles in determining the stability and accuracy of the cycling system.

Another important aspect of cycling is the accumulation of information over time. Even when observations are sparse at a given analysis step, the model dynamics can transport observational information forward, allowing past observations to influence future states. This temporal coupling is one of the key strengths of cycling data assimilation systems, enabling improved state estimation beyond what is possible from a single analysis.

This tutorial introduces the implementation of a cycling LETKF system, focusing on both the algorithmic structure and the practical considerations required for stable performance. We begin by outlining the forecast–analysis cycle and its role in propagating ensemble-based uncertainty. We examine the impact of removing inflation within a cycling context, highlighting how these factors influence long-term system behavior. Finally, we explore diagnostic tools, such as ensemble spread and RMSE, to evaluate the performance of the cycling system over multiple assimilation cycles in so-called **sawtooth diagrams**

### Set Environment Variables and Number of Cycles

In this tutorial, in addition to setting the typical JEDI_BIN and JEDI_EDU variables, we will automate how many cycles are performed using the global variable `ncycles`. By default this is set to 6. 
> NOTE: **If you would like to perform more than 6 cycles, you will need to first lengthen the truth simulation in 0.qg_tutorial_start**. This can be accomplished by opening `qgstart/yamls/generate_truth.yaml`, editing `forecast length: P21D` to something longer, then saving it and rerunning the truth simulation within the **0.qg_tutorial_start** tutorial

In [ ]:
export JEDI_BIN=/home/nonroot/build/bin
export JEDI_EDU=/home/nonroot/shared/EDU
export ncycles=6 # If you want more than 6, you need to extend the truth simulation in 0.qg_tutorial_start!!!

# Experiment 1: LETKF Cycling with multiple observations
***

For this cycling experiment, we will use 100 randomly generated observations for each cycle time, to mimic the sort of cycling performed with real observations in operational NWP centers.

### Make Observations for All Cycles
Recall the `makeobs3d_mult_obs.yaml` yaml file from the `qgstart` tutorial:

```yaml
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
state:
  date: 2009-12-31T00:00:00Z
  filename: qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P16D.nc
window begin: 2009-12-30T18:00:00Z
window length: PT12H
observations:
  observers:
  - obs operator:
      obs type: Stream              # The observation type is the stream function
    obs space:
      obsdataout:
        engine:
          obsfile: qgstart/output/obs/truth_100obs.obs3d.nc
      obs type: Stream
      generate:
        begin: PT6H
        nval: 1
        obs density: 100            # Generate 100 observations at random locations
        obs error: 4.0e6            # Observation error standard deviation, in m^2/s
        obs period: PT12H
make obs: true                      # Generate the observations
obs perturbations: true             # Add random  measurements to the observations
```

We will now need to generate observations at several cycles. Given a cycle length of 24 hours, we will need yaml files for every day from this date above (2009-12-31) until the last cycle (2010-01-05 for `ncycles=6`). This can be done dynamically within a loop.

For the above yaml, what parts will need to be modified for each cycle? Try to think on where you would modify the file before reading below.

The answer is the following
- **date:** Must be incremented for each day
- **filename:** The "P16D" will also need to be incremented for each successive day, to generate obs from different days of the truth
- **window begin:** Must be consistent with date and window length (minus 6 hours of date in this case)
- **obsfile:** create unique obsfile names for different cycles

One tricky aspect of this is how to loop dates. Fortunately, the bash command `date` can help in this instance, where you give it the original date and an increment (in days or hours) and it can output the adjusted date in the format specified.

Take a look at how the loop is structured below. Pay attention to how `curdate` is constructed for each cycle and advanced at the end of the loop to the next cycle, how `curdate_minus6h` is created off of curdate, and how the `forecast_hour` is created using simple bash math addition inside the $((...)) syntax. The `echo`command prints out the results of each variable for each cycle. Confirm this is what you expect.

The `cat` command is one option to allow us to dynamically enter this info into unique yaml files for each cycle. Each file is saved at `./qgcycling/yamls/makeobs3d_mult_obs_cyc${icyc}.yaml` for each cycle (icyc = 1,2,3,4,5,6)


In [ ]:
cd $JEDI_EDU

firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   curdate_minus6h=$(date -u -d "$curdate -6 hours" +"%Y-%m-%dT%H:%M:%SZ")
   forecast_hour="P$((icyc+15))D"
   echo Cycle $icyc: curdate = $curdate, window begin = $curdate_minus6h, forecast time = $forecast_hour
   
cat << EOF > ./qgcycling/yamls/makeobs3d_mult_obs_cyc${icyc}.yaml
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
state:
  date: ${curdate}
  filename: qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.${forecast_hour}.nc
window begin: ${curdate_minus6h}
window length: PT12H
observations:
  observers:
  - obs operator:
      obs type: Stream                             # The observation type is the stream function
    obs space:
      obsdataout:
        engine:
          obsfile: qgstart/output/obs/truth_cyc${icyc}.obs3d.nc
      obs type: Stream
      generate:
        begin: PT6H
        nval: 1
        obs density: 100                           # Generate 100 observations at random locations
        obs error: 4.0e6                           # Observation error standard deviation, in m^2/s
        obs period: PT12H
make obs: true                                     # Generate the observations
obs perturbations: true                            # Add random  measurements to the observations
EOF

   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

After running, you can verify that there were yaml files created. There should be as many files as there are cycles (set by `ncycles`)

In [ ]:
cd $JEDI_EDU
ls ./qgcycling/yamls/makeobs3d_mult_obs_cyc*yaml

We can now run the `qg_hofx` program as in the qgstart tutorial to create observations for each cycle. Note that the outputted cycle observations are saved in the `qgstart/output/obs` directory, as in the qgstart tutorial. For efficiency, the output of the JEDI program has been redirected to log files, with a simple echo statement to confirm each cycle has successfully run or encountered an error.

In [ ]:
cd $JEDI_EDU
for icyc in $(seq 1 $ncycles); do
    $JEDI_BIN/qg_hofx3d.x ./qgcycling/yamls/makeobs3d_mult_obs_cyc${icyc}.yaml >& ./qgcycling/output/makeobs_cyc${icyc}.log
    if [ $? -ne 0 ]; then
        echo "ERROR! Something went wrong running for cycle $icyc. Check ./qgcycling/output/makeobs_cyc${icyc}.log"
    else
        echo "SUCCESS run for cycle $icyc"
    fi
done

If you encounter any issues, check the output log created. Otherwise, let's quickly verify observations were created for all cycles with an `ls` command:

In [ ]:
cd $JEDI_EDU
ls  ./qgstart/output/obs/truth_cyc*.obs3d.nc

### Construct LETKF and forecast yamls

Look at the template file `qgcycling/yamls/letkf_template.yaml`. We will use another method besides `cat` to construct yaml files for each cycle, using a find and replace of certain template keywords in the letkf_template.yaml file (in this case, capitalized words starting with @ are the keywords to be searched and replaced, such as @DATE_BGN). The bash command `sed` can be used for this purpose. Notice we use the same looping with dates and cycles as in the above for observation generation.

> <u>A note on `bgfile`</u>: In cycles after the first, we use output produced from the `qg_forecast.x` program as the background files. Thus, this name **must** be consistent with the output names defined in the  yaml files for the forecast (see `forecast_template.yaml` to confirm this consistency)

In [ ]:
cd $JEDI_EDU
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   # for window begin:
   curdate_minus6h=$(date -u -d "$curdate -6 hours" +"%Y-%m-%dT%H:%M:%SZ")
   # for bgfile:
   curdate_minus1day=$(date -u -d "$curdate -24 hours" +"%Y-%m-%dT%H:%M:%SZ")

   if [ $icyc -eq 1 ]; then
      bgfile="qgstart/output/bg/bkgd.ens.%mem%.${curdate_minus1day}.P1D.nc"
   else
      bgfile="qgcycling/output/exp_letkf/da/letkf.%mem%.fc.${curdate_minus1day}.P1D.nc"
   fi

   echo Cycle $icyc: curdate = $curdate, window begin = $curdate_minus6h, bgfile = $bgfile
   
   sed -e "s|@DATE_BGN|${curdate_minus6h}|g;s|@DATE_ANL|${curdate}|g" \
       -e "s|@CYC|$icyc|g;s|@BACKGROUND_FILE|${bgfile}|g" \
       ./qgcycling/yamls/letkf_template.yaml > ./qgcycling/yamls/letkf_cyc${icyc}.yaml

   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

Next, we will generate forecast yamls for each cycle and each ensemble member. In this case, a nested loop is required. See `forecast_template.yaml` for the template of each cycle and member. Since all forecasts come after an analysis is produced, the names of the initial condition (IC) files are all the same, only different by what date the ICs are at. Thuse we have only to replace the ensemble member and IC date template keywords with the looped variables for each.

In [ ]:
cd $JEDI_EDU
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   for imem in $(seq 1 20); do
      sed -e "s|%MEMBER%|$(printf "%06d\n" "$imem")|g;s|%MEM%|$imem|g;s|@DATE_IC|${curdate}|g" \
             ./qgcycling/yamls/forecast_template.yaml > ./qgcycling/yamls/forecast_cyc${icyc}_mem${imem}.yaml
   done 

    echo "Cycle $icyc: curdate = $curdate, forecast_cyc${icyc}_mem*.yaml files created!"

   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done
  

### Run the cycling!

Everything has been produced now to run the cycling experiments fully. Now we loop one more time through the cycles, running first the LETKF step, followed by the ensemble forecast step using the LETKF analysis. This is all setup from the above yaml files that were created. Try to get an understanding of how all it all connects together before running.

This is very similar in concept to what real operational NWP centers do with their DA cycling setups (though in their case, only one loop is generally done, and namelist/yaml files are created on-the-fly rather than pre-generated as we have done here).

NOTE for 6 cycles this will take at least 2 minutes to complete in total, possibly longer depending on the speed of your local machine.

In [ ]:
cd $JEDI_EDU
for icyc in $(seq 1 $ncycles); do
   start_time=`date +%s` # Needed to calculate time it takes to complete each step
   
   # First run LETKF
   $JEDI_BIN/qg_letkf.x ./qgcycling/yamls/letkf_cyc${icyc}.yaml >& ./qgcycling/output/letkf_cyc${icyc}.log
   if [ $? -ne 0 ]; then
      echo "ERROR! Something went wrong running LETKF for cycle $icyc. Check ./qgcycling/output/letkf_cyc${icyc}.log"
   else
      end_time_letkf=`date +%s`
      echo "SUCCESS LETKF run for cycle $icyc, took $(( end_time_letkf - start_time )) seconds"
   fi
   
   # Then run QG ensemble forecast by looping over each member 
   for imem in $(seq 1 20); do
      $JEDI_BIN/qg_forecast.x ./qgcycling/yamls/forecast_cyc${icyc}_mem${imem}.yaml >& ./qgcycling/output/exp_letkf/forecast_cyc${icyc}_mem${imem}.log
      if [ $? -ne 0 ]; then
         echo "ERROR! Something went wrong running forecast for cycle $icyc, member $imem. Check ./qgcycling/output/exp_letkf/forecast_cyc${icyc}_mem${imem}.log"
      else
         echo "SUCCESS forecast run for cycle $icyc, member $imem"
      fi
   done
   end_time_fcst=`date +%s`
   echo "SUCCESS ensemble forecast run for cycle $icyc, took $(( end_time_fcst - end_time_letkf )) seconds"

done

Let's look at some of the results produced within the `qgcycling/output/exp_letkf/da` directory. We will just list out contents from member 1, but you should confirm that 20 members of output were produced in the aforementioned directory.

In [ ]:
cd $JEDI_EDU
echo "Ensemble Member 1 Analyses (used as ICs for member 1 forecast to the next cycle):"
ls ./qgcycling/output/exp_letkf/da/letkf.000001.an*nc
echo -e "\nEnsemble Member 1 Forecasts (used as backgrounds for each cycle):"
ls ./qgcycling/output/exp_letkf/da/letkf.1.fc*P1D.nc
echo -e "\nEnsemble Mean Analyses:"
ls ./qgcycling/output/exp_letkf/da/letkf.000000.an*nc
echo -e "\nEnsemble Mean Prior and Increments:"
ls ./qgcycling/output/exp_letkf/da/letkf_mean*nc


### Compute RMSE, Plot Sawtooths 
We could, at this point, plot increments, analysis errors, forecast errors, etc. for each cycle. However, it is difficult to get a sense for the overall performance of the LETKF assimilation based off of these alone. In this case, we want to quantify the performance of each cycle and plot them to keep track of the stability of the DA system over time. For this purpose, **we will compute RMSE against the truth for both ensemble mean background and mean analysis files at each cycle**. The python script `compute_rmse_layers.py` has been provided for this purpose. To run this script, you need to provide it the model file to evaluate (either mean prior or mean analysis), the truth file to verify against, which variable to compute RMSE for (x,u,v,q), and the destination text file to output the results into, i.e.,

> python compute_rmse_layers.py <input_model.nc> <input_truth.nc> -v x -o <output_rmse_file.txt> -l \<line_label_in_output_text_file>

In [ ]:
cd $JEDI_EDU
output_filename="./qgcycling/output/exp_letkf/plots/rmse_stream.txt"

# Remove text file if it exists
[[ -f "$output_filename" ]] && rm -v $output_filename

firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   echo "Computing rmse for $curdate . . ."
   # Run for background mean file
   python ./plots_scripts/compute_rmse_layers.py \
          ./qgcycling/output/exp_letkf/da/letkf_meanprior.an.${curdate}.nc \
          ./qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P$((icyc+15))D.nc \
          -v x -o $output_filename -l "cyc${icyc} bg"

   # Run for analysis mean file
   python ./plots_scripts/compute_rmse_layers.py \
          ./qgcycling/output/exp_letkf/da/letkf.000000.an.${curdate}.nc \
          ./qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P$((icyc+15))D.nc \
          --v x -o $output_filename -l "cyc${icyc} an"
   
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

Next, the python program `plot_rmse_sawtooth.py` reads in the  rmse output file produced by the loop above, parses the information, and plots the sawtooth of background and analysis RMSE for each cycle

Usage:
> python plot_rmse_sawtooth.py <rmse_file.txt> -o <name_of_sawtooth_file.png>

In [ ]:
cd $JEDI_EDU/plots_scripts

python plot_rmse_sawtooth.py $JEDI_EDU/qgcycling/output/exp_letkf/plots/rmse_stream.txt \
                             -o $JEDI_EDU/qgcycling/output/exp_letkf/plots/rmse_sawtooth.png


<center><img src="images/rmse_sawtooth.png" alt="sawtooth" width="600"/></center>

Another python script, `compute_ensvar_layers.py`, is provided to compute the 2d-average of ensemble variance for each layer, and output to a text file. It is setup similar to the RMSE computation script.

### Compute Ensemble Variance
To fully assess the performance of LETKF, we need to compute the ensemble variance and compare it to the RMSE values. Since the ensemble variance is a measure of the forecast uncertainty, in a well-tuned system the mean background error should match the estimated uncertainty (ensemble spread, i.e. square root of ensemble variance) from the ensemble. So we will use the 

In [ ]:
cd $JEDI_EDU/
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
 for anorbg in bg an; do
   curdate_minus1day=$(date -u -d "$curdate -24 hours" +"%Y-%m-%dT%H:%M:%SZ")
   if [ "$anorbg" == "bg" ]; then
      zpad=0
      if [ $icyc -eq 1 ]; then
         bgfile_mem1="qgstart/output/bg/bkgd.ens.1.${curdate_minus1day}.P1D.nc"
         bgfile="qgstart/output/bg/bkgd.ens.%mem%.${curdate_minus1day}.P1D.nc"
      else
         bgfile_mem1="qgcycling/output/exp_letkf/da/letkf.1.fc.${curdate_minus1day}.P1D.nc"
         bgfile="qgcycling/output/exp_letkf/da/letkf.%mem%.fc.${curdate_minus1day}.P1D.nc"
      fi
   else
       zpad=6
       bgfile_mem1="qgcycling/output/exp_letkf/da/letkf.000000.an.${curdate}.nc"
       bgfile="qgcycling/output/exp_letkf/da/letkf.%mem%.an.${curdate}.nc"
   fi

   # Build namelist for ens variance JEDI program
   cat << EOF > ./qgcycling/yamls/ens_variance_cyc${icyc}.yaml
background:
  date: $curdate
  filename: $bgfile_mem1
  state variables: [x,q,u,v]
ensemble:
  members from template:
    template:
      date: $curdate
      filename: $bgfile
      state variables: [x,q,u,v]
    pattern: %mem%
    zero padding: $zpad
    nmembers: 20
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
variance output:
  datadir:  qgcycling/output/exp_letkf/da
  exp: ${anorbg}EnsVariance
  type: diag
  date: $curdate
EOF
   # Run the ens variance program
   $JEDI_BIN/qg_ens_variance.x ./qgcycling/yamls/ens_variance_cyc${icyc}.yaml \
                            >& ./qgcycling/output/exp_letkf/ens_var_cyc${icyc}_$anorbg.log

   if [ $? -ne 0 ]; then
      echo "ERROR! Something went wrong running qg_ens_variance.x for cycle $icyc. Check ./qgcycling/output/exp_letkf/ens_var_cyc${icyc}_$anorbg.log"
   else
      echo "SUCCESS qg_ens_variance.x run for cycle $icyc $anorbg"
   fi
 done #anorbg loop
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done #icyc loop

ls -lhtr ./qgcycling/output/exp_letkf/da/*EnsVariance*nc

In [ ]:
cd $JEDI_EDU
output_filename="./qgcycling/output/exp_letkf/plots/ensvar_stream.txt"
# Remove text file if it exists
[[ -f "$output_filename" ]] && rm -v $output_filename

firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   echo "Computing average ensemble variance for $curdate . . ."
   # Run for background mean file
   python ./plots_scripts/compute_ensvar_layers.py \
          ./qgcycling/output/exp_letkf/da/bgEnsVariance.diag.${curdate}.nc \
          -v x -o $output_filename -l "cyc${icyc} bg"

   python ./plots_scripts/compute_ensvar_layers.py \
          ./qgcycling/output/exp_letkf/da/anEnsVariance.diag.${curdate}.nc \
          -v x -o $output_filename -l "cyc${icyc} an"
   
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

The `plot_rmse_sawtooth_withvar.py` script has been provided to plot the sawtooth with ensemble spread overlaid

In [ ]:
cd $JEDI_EDU/plots_scripts

python plot_rmse_sawtooth_withvar.py -m 4e7 $JEDI_EDU/qgcycling/output/exp_letkf/plots/rmse_stream.txt \
                             $JEDI_EDU/qgcycling/output/exp_letkf/plots/ensvar_stream.txt \
                             -o $JEDI_EDU/qgcycling/output/exp_letkf/plots/rmse_sawtooth_withvar.png


<center><img src="images/rmse_sawtooth_withvar.png" alt="sawtooth_withspread" width="600"/></center>

Now that ensemble spread is plotted along with the RMSE, try to answer these questions:
- Is the system well-tuned? Why or why not?
- What does ensemble spread below the first guess RMSE indicate?

The second question is an important consideration of ensemble-based DA diagnostics. Ideally, spread and RMSE of the first guess should match. In this case, the ensemble spread is below the first guess RMSE, indicative of so-called **underdispersion**. This term means that the ensemble variance itself does not represent the true uncertainty/error of the first guess forecast. (Conversely, overdispersion occurs when ensemble variance is too high compared to first guess mean error, so we overestimate the uncertainty using the ensemble)

Although this is a good starting point, underdispersion of the last 4 cycles indicates more should be done to tune the system for better performance: optimizing localization scale, further tuning the covariance inflations, perhaps even changing the number of ensemble members. The capstone project will explore how to perform some of these tunings, and each time we will examine these statistics to evaluate how to best set these values. Perhaps we can get lower RMSE of first guess forecasts with better-tuned localization, or perhaps the analysis ensemble inflation can help to tune the ensemble forecast spread to better match the true error distribution in later cycles.

Next here, we will explore the effect of covariance inflation on cycling. Specifically, we will turn off covariance inflation and see what happens.


# Experiment 2: Cycling Without Covariance Inflation
***

For this cycling experiment, we will test what happens if we remove **covariance inflation**. Finite ensembles tend to underestimate forecast uncertainty, causing the ensemble spread to collapse over time as each ensemble analysis reduces the spread further.  To counteract this effect, covariance inflation is applied.

In experiment 1, two covariance methods were employed:  **Multiplicative Inflation** and **Relaxation to Prior Perturbations (RTPP)** . 
<center><u><b>Multiplicative Inflation</b></u></center>
<center>${\mathbf{X}^a_{i,\text{Inf}}}​→r{\mathbf{X}^a_{i}}​,r>1$</center> 
<br><center><u><b>Relaxation to Prior Perturbations (RTPP)</b></u></center>
<center>${\mathbf{X}^a_{i,\text{Inf}}} = \alpha {\mathbf{X}^b} + (1 - \alpha) {\mathbf{X}^a_i}$</center>

Note the parameters for each method are different. In multiplicative inflation, $r$ is chosen to be some value above 1, applied directly to the ensemble analysis perturbations. In our default cycling experiments this has been set to 1.1, or 10% inflation. Common values used fall in the 0-20% range (1.0 to 1.2) 

In RTPP, α is a relaxation or blending parameter, some value between 0 and 1. As α increases, RTPP increasingly blends more from the prior ensemble perturbations ${\mathbf{X}^b}$. Most common values tend to fall around 0.5-0.9. 

But it should be noted, proper choices of inflation are highly dependent on the DA system. Those value ranges should be used as a starting reference, but each system should be tuned to optimize the settings, such that ensemble spread matches the mean error as best as possible. 

In this cycling experiment, we will examine what happens when covariance inflation is turned off. Take a look at `letkf_template_noinflation.yaml`, in comparison with `letkf_template.yaml`. The main difference is pasted below<br><br>

<u>`letkf_template.yaml`</u>
```yaml
local ensemble DA:
  solver: LETKF
  inflation:
    rtpp: 0.5
    mult: 1.1
```
<u>`letkf_template_noinflation.yaml`</u>
```yaml
local ensemble DA:
  solver: LETKF
  inflation:
    rtpp: 0.0
    mult: 1.0
```

So we have effectively turned off inflation by setting the inflation parameters $\alpha$  and $r$  to 0 and 1, respectively.

### Construct LETKF and forecast yamls

The only change from experiment 1 is to the inflation, set in  `qgcycling/yamls/letkf_template_noinflation.yaml`. We also will output the results to a new directory `exp_letkf_noinf`. These changes are reflected in the letkf template yaml as well as the forecast template yaml `qgcycling/yamls/forecast_template_noinflation.yaml`

In [ ]:
cd $JEDI_EDU
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   # for window begin:
   curdate_minus6h=$(date -u -d "$curdate -6 hours" +"%Y-%m-%dT%H:%M:%SZ")
   # for bgfile:
   curdate_minus1day=$(date -u -d "$curdate -24 hours" +"%Y-%m-%dT%H:%M:%SZ")

   if [ $icyc -eq 1 ]; then
      bgfile="qgstart/output/bg/bkgd.ens.%mem%.${curdate_minus1day}.P1D.nc"
   else
      bgfile="qgcycling/output/exp_letkf_noinf/da/letkf.%mem%.fc.${curdate_minus1day}.P1D.nc"
   fi

   echo Cycle $icyc: curdate = $curdate, window begin = $curdate_minus6h, bgfile = $bgfile
   
   sed -e "s|@DATE_BGN|${curdate_minus6h}|g;s|@DATE_ANL|${curdate}|g" \
       -e "s|@CYC|$icyc|g;s|@BACKGROUND_FILE|${bgfile}|g" \
       ./qgcycling/yamls/letkf_template_noinflation.yaml > ./qgcycling/yamls/letkf_cyc${icyc}_noinf.yaml

   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

Next, we will generate forecast yamls for each cycle and each ensemble member. In this case, a nested loop is required. See `forecast_template.yaml` for the template of each cycle and member. Since all forecasts come after an analysis is produced, the names of the initial condition (IC) files are all the same, only different by what date the ICs are at. Thuse we have only to replace the ensemble member and IC date template keywords with the looped variables for each.

In [ ]:
cd $JEDI_EDU
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   for imem in $(seq 1 20); do
      sed -e "s|%MEMBER%|$(printf "%06d\n" "$imem")|g;s|%MEM%|$imem|g;s|@DATE_IC|${curdate}|g" \
             ./qgcycling/yamls/forecast_template_noinflation.yaml > ./qgcycling/yamls/forecast_cyc${icyc}_mem${imem}_noinf.yaml
   done 

    echo "Cycle $icyc: curdate = $curdate, forecast_cyc${icyc}_mem*.yaml files created!"

   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done
  

### Run the cycling!

Everything has been produced now to run the cycling experiments fully. Now we loop one more time through the cycles, running first the LETKF step, followed by the ensemble forecast step using the LETKF analysis. This is all setup from the above yaml files that were created. .

NOTE for 6 cycles this will take at least 2 minutes to complete in total, possibly longer depending on the speed of your local machine.

In [ ]:
cd $JEDI_EDU
for icyc in $(seq 1 $ncycles); do
   start_time=`date +%s` # Needed to calculate time it takes to complete each step
   
   # First run LETKF
   $JEDI_BIN/qg_letkf.x ./qgcycling/yamls/letkf_cyc${icyc}_noinf.yaml >& ./qgcycling/output/letkf_cyc${icyc}_noinf.log
   if [ $? -ne 0 ]; then
      echo "ERROR! Something went wrong running LETKF for cycle $icyc. Check ./qgcycling/output/letkf_cyc${icyc}_noinf.log"
      break 
   else
      end_time_letkf=`date +%s`
      echo "SUCCESS LETKF run for cycle $icyc, took $(( end_time_letkf - start_time )) seconds"
   fi
   
   # Then run QG ensemble forecast by looping over each member 
   for imem in $(seq 1 20); do
      $JEDI_BIN/qg_forecast.x ./qgcycling/yamls/forecast_cyc${icyc}_mem${imem}_noinf.yaml >& ./qgcycling/output/exp_letkf/forecast_cyc${icyc}_mem${imem}_noinf.log
      if [ $? -ne 0 ]; then
         echo "ERROR! Something went wrong running forecast for cycle $icyc, member $imem. Check ./qgcycling/output/exp_letkf/forecast_cyc${icyc}_mem${imem}_noinf.log"
         break 2
      else
         echo "SUCCESS forecast run for cycle $icyc, member $imem"
      fi
   done
   end_time_fcst=`date +%s`
   echo "SUCCESS ensemble forecast run for cycle $icyc, took $(( end_time_fcst - end_time_letkf )) seconds"

done

Let's look at some of the results produced within the `qgcycling/output/exp_letkf/da` directory. We will just list out contents from member 1, but you should confirm that 20 members of output were produced in the aforementioned directory.

In [ ]:
cd $JEDI_EDU
echo "Ensemble Member 1 Analyses (used as ICs for member 1 forecast to the next cycle):"
ls ./qgcycling/output/exp_letkf_noinf/da/letkf.000001.an*nc
echo -e "\nEnsemble Member 1 Forecasts (used as backgrounds for each cycle):"
ls ./qgcycling/output/exp_letkf_noinf/da/letkf.1.fc*P1D.nc
echo -e "\nEnsemble Mean Analyses:"
ls ./qgcycling/output/exp_letkf_noinf/da/letkf.000000.an*nc
echo -e "\nEnsemble Mean Prior and Increments:"
ls ./qgcycling/output/exp_letkf_noinf/da/letkf_mean*nc


### Compute RMSE, Plot Sawtooths 

> python compute_rmse_layers.py <input_model.nc> <input_truth.nc> -v x -o <output_rmse_file.txt> -l \<line_label_in_output_text_file>

In [ ]:
cd $JEDI_EDU
output_filename="./qgcycling/output/exp_letkf_noinf/plots/rmse_stream.txt"

# Remove text file if it exists
[[ -f "$output_filename" ]] && rm -v $output_filename

firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   echo "Computing rmse for $curdate . . ."
   # Run for background mean file
   python ./plots_scripts/compute_rmse_layers.py \
          ./qgcycling/output/exp_letkf_noinf/da/letkf_meanprior.an.${curdate}.nc \
          ./qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P$((icyc+15))D.nc \
          -v x -o $output_filename -l "cyc${icyc} bg"

   # Run for analysis mean file
   python ./plots_scripts/compute_rmse_layers.py \
          ./qgcycling/output/exp_letkf_noinf/da/letkf.000000.an.${curdate}.nc \
          ./qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P$((icyc+15))D.nc \
          --v x -o $output_filename -l "cyc${icyc} an"
   
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

Next, the python program `plot_rmse_sawtooth.py` reads in the  rmse output file produced by the loop above, parses the information, and plots the sawtooth of background and analysis RMSE for each cycle

Usage:
> python plot_rmse_sawtooth.py <rmse_file.txt> -o <name_of_sawtooth_file.png>

In [ ]:
cd $JEDI_EDU/plots_scripts

python plot_rmse_sawtooth.py $JEDI_EDU/qgcycling/output/exp_letkf_noinf/plots/rmse_stream.txt \
                             -o $JEDI_EDU/qgcycling/output/exp_letkf_noinf/plots/rmse_sawtooth.png

<center><img src="images/Sawtooth_noinf.png" alt="sawtooth_noinf" width="600"/></center>

### Compute Ensemble Variance, Plot on Sawtooth
To fully assess the performance of LETKF, we need to compute the ensemble variance and compare it to the RMSE values. Since the ensemble variance is a measure of the forecast uncertainty, in a well-tuned system the mean background error should match the estimated uncertainty (ensemble spread, i.e. square root of ensemble variance) from the ensemble. 

In [ ]:
cd $JEDI_EDU/
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
 for anorbg in bg an; do
   curdate_minus1day=$(date -u -d "$curdate -24 hours" +"%Y-%m-%dT%H:%M:%SZ")
   if [ "$anorbg" == "bg" ]; then
      zpad=0
      if [ $icyc -eq 1 ]; then
         bgfile_mem1="qgstart/output/bg/bkgd.ens.1.${curdate_minus1day}.P1D.nc"
         bgfile="qgstart/output/bg/bkgd.ens.%mem%.${curdate_minus1day}.P1D.nc"
      else
         bgfile_mem1="qgcycling/output/exp_letkf_noinf/da/letkf.1.fc.${curdate_minus1day}.P1D.nc"
         bgfile="qgcycling/output/exp_letkf_noinf/da/letkf.%mem%.fc.${curdate_minus1day}.P1D.nc"
      fi
   else
       zpad=6
       bgfile_mem1="qgcycling/output/exp_letkf_noinf/da/letkf.000000.an.${curdate}.nc"
       bgfile="qgcycling/output/exp_letkf_noinf/da/letkf.%mem%.an.${curdate}.nc"
   fi

   # Build namelist for ens variance JEDI program
   cat << EOF > ./qgcycling/yamls/ens_variance_cyc${icyc}_noinf.yaml
background:
  date: $curdate
  filename: $bgfile_mem1
  state variables: [x,q,u,v]
ensemble:
  members from template:
    template:
      date: $curdate
      filename: $bgfile
      state variables: [x,q,u,v]
    pattern: %mem%
    zero padding: $zpad
    nmembers: 20
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
variance output:
  datadir:  qgcycling/output/exp_letkf_noinf/da
  exp: ${anorbg}EnsVariance
  type: diag
  date: $curdate
EOF
   # Run the ens variance program
   $JEDI_BIN/qg_ens_variance.x ./qgcycling/yamls/ens_variance_cyc${icyc}_noinf.yaml \
                            >& ./qgcycling/output/exp_letkf/ens_var_cyc${icyc}_${anorbg}_noinf.log

   if [ $? -ne 0 ]; then
      echo "ERROR! Something went wrong running qg_ens_variance.x for cycle $icyc. Check ./qgcycling/output/exp_letkf/ens_var_cyc${icyc}_${anorbg}_noinf.log"
      break
   else
      echo "SUCCESS qg_ens_variance.x run for cycle $icyc $anorbg"
   fi
 done #anorbg loop
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done #icyc loop

ls -lhtr ./qgcycling/output/exp_letkf_noinf/da/*EnsVariance*nc

In [ ]:
cd $JEDI_EDU
output_filename="./qgcycling/output/exp_letkf_noinf/plots/ensvar_stream.txt"
# Remove text file if it exists
[[ -f "$output_filename" ]] && rm -v $output_filename

firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   echo "Computing average ensemble variance for $curdate . . ."
   # Run for background mean file
   python ./plots_scripts/compute_ensvar_layers.py \
          ./qgcycling/output/exp_letkf_noinf/da/bgEnsVariance.diag.${curdate}.nc \
          -v x -o $output_filename -l "cyc${icyc} bg"

   python ./plots_scripts/compute_ensvar_layers.py \
          ./qgcycling/output/exp_letkf_noinf/da/anEnsVariance.diag.${curdate}.nc \
          -v x -o $output_filename -l "cyc${icyc} an"
   
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

The `plot_rmse_sawtooth_withvar.py` script has been provided to plot the sawtooth with ensemble spread overlaid

In [ ]:
cd $JEDI_EDU/plots_scripts

python plot_rmse_sawtooth_withvar.py -m 4e7 $JEDI_EDU/qgcycling/output/exp_letkf_noinf/plots/rmse_stream.txt \
                             $JEDI_EDU/qgcycling/output/exp_letkf_noinf/plots/ensvar_stream.txt \
                             -o $JEDI_EDU/qgcycling/output/exp_letkf_noinf/plots/rmse_sawtooth_withvar.png


<center><img src="images/Sawtooth_compare.png" alt="sawtooth_compare" width="600"/></center>

- Is the system still stable? What changed?
- Are there any signs of filter divergence? How can you tell?

Note that while for these six cycles everything ran to completion, we can see the system is not well-tuned. Though filter divergence was not encountered yet with these 6 cycles, we can see the spread is reducing with successive cycles, coupled with increasing error and less fit to the observations  in the ensemble mean at later cycles. Further cycling from this point will not be reliable, as ensembles that are underdispersed (or overconfident) run the risk of "spread collapse" and filter divergence. 

## Summary

In this tutorial, we explored the behavior of a cycling data assimilation system using the LETKF. Through a baseline experiment with a standard configuration, we demonstrated how the forecast–analysis cycle evolves over time, with each analysis improving the background state for the next cycle. The resulting sawtooth patterns in RMSE  illustrate the fundamental structure of cycling systems: forecast error grows during the model integration and is reduced when observations are assimilated at each analysis step.

A second experiment examined the role of covariance inflation by removing it from the cycling system. Without inflation, the ensemble spread progressively decreased, leading to an underestimation of forecast uncertainty. This caused the filter to become overconfident, reducing the impact of observations and degrading overall performance. In the sawtooth diagnostics, this behavior is reflected in diminishing analysis improvements and, in some cases, increasing forecast error over successive cycles.

Together, these experiments highlight two key principles of ensemble-based data assimilation. First, cycling allows information from observations to accumulate over time, improving state estimates beyond a single analysis. Second, maintaining a realistic representation of uncertainty—through techniques such as covariance inflation—is essential for stable and accurate performance. These concepts are central to the successful application of LETKF and other ensemble-based methods in atmospheric data assimilation systems.

The Capstone project notebook `qg_cycling_capstone.ipynb` expands upon these ideas, leaving **you** to be the researcher trying to tune your very own DA and forecast cycling system. It uses a reduced resolution ensemble, making the problem more challenging to tune effectively. This will provide hands on experience to see how changes to these parameters affect the results. 